# Experiment: PF-06409577 Probe Material Gate

Objective:
- verify whether PF-06409577 is defined enough for an `AMPKB1` expansion-lane quote addendum;
- keep chemistry identity, target selectivity, HbF bridge evidence, and safety boundary separate;
- reject any patient-facing or generic AMPK/NRF2 framing.

Done means the notebook emits a compact decision object that can be copied into a finding and lab-quote brief.

In [1]:
from __future__ import annotations

import json
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path

ROOT = Path.cwd()

PUBCHEM_PROPERTIES_PATH = (
    ROOT / "data/chemistry/pubchem/ampk-probes/2026-04-27-pf-06409577-properties.json"
)
PUBCHEM_DESCRIPTION_PATH = (
    ROOT / "data/chemistry/pubchem/ampk-probes/2026-04-27-pf-06409577-description.json"
)
CHEMBL_SEARCH_PATH = (
    ROOT / "data/chemistry/chembl/ampk-probes/2026-04-27-pf-06409577-search.json"
)
CHEMBL_ACTIVITIES_PATH = (
    ROOT
    / "data/chemistry/chembl/ampk-probes/2026-04-27-pf-06409577-chembl-activities.json"
)
PUBMED_BETA1_PATH = (
    ROOT / "data/literature/pubmed/2026-04-27-pf-06409577-ampk-beta1-search.json"
)
PUBMED_HBF_REFRESH_PATH = (
    ROOT / "data/literature/pubmed/2026-04-27-expansion-ampkb1-nrf2-ulk1-hbf.json"
)
AMPKB1_ABSTRACT_PATH = (
    ROOT / "data/literature/pubmed/2026-04-27-ampkb1-hbf-nrf2-abstract.xml"
)

## Evidence Inputs

This gate uses source snapshots already retrieved into the repo:

- PubChem identity for CID `71748255`;
- ChEMBL identity and activities for `CHEMBL3891221`;
- PubMed snapshots for PF-06409577 / AMPK beta1 and AMPK / HbF / NRF2.

The gate is intentionally conservative: identity and assay readiness can pass while patient treatment remains blocked.

In [2]:
@dataclass(frozen=True)
class ChemicalIdentity:
    """Chemical identity fields needed before asking a lab to source material."""

    cid: int
    title: str
    chembl_id: str
    formula: str
    chembl_formula: str
    molecular_weight: str
    inchikey: str


@dataclass(frozen=True)
class ActivitySummary:
    """ChEMBL activity summary for beta1 selectivity gating."""

    beta1_nanometer_values: tuple[float, ...]
    beta2_floor_values: tuple[float, ...]

    @property
    def has_beta1_activity(self) -> bool:
        """Return whether beta1-complex activity is in the nanomolar range."""
        return any(value <= 30 for value in self.beta1_nanometer_values)

    @property
    def has_beta2_selectivity_floor(self) -> bool:
        """Return whether beta2-complex activity is weak above the gate floor."""
        return any(value >= 40_000 for value in self.beta2_floor_values)


@dataclass(frozen=True)
class LiteratureSummary:
    """Literature anchors and cautions for PF-06409577."""

    has_hbf_bridge: bool
    has_nrf2_bridge: bool
    has_ulk1_p62_bridge: bool
    has_ugt1a1_caution: bool
    pmids: tuple[str, ...]


@dataclass(frozen=True)
class MaterialGateResult:
    """Final material-gate result for the AMPKB1 expansion lane."""

    identity_verified: bool
    beta1_selectivity_supported: bool
    hbf_mechanism_bridge_supported: bool
    metabolism_caution_present: bool
    decision: str
    boundary: str

In [3]:
def load_json(path: Path) -> dict[str, object]:
    """Load a JSON source snapshot as a mapping."""
    return json.loads(path.read_text())


def first_item(value: object) -> dict[str, object]:
    """Return the first mapping from a source-list value."""
    if isinstance(value, list) and value and isinstance(value[0], dict):
        return value[0]
    return {}


def as_float(value: object) -> float | None:
    """Convert a JSON scalar to float when possible."""
    if isinstance(value, int | float | str):
        try:
            return float(value)
        except ValueError:
            return None
    return None


def build_identity() -> ChemicalIdentity:
    """Build the cross-database identity packet."""
    pubchem_properties = load_json(PUBCHEM_PROPERTIES_PATH)
    pubchem_description = load_json(PUBCHEM_DESCRIPTION_PATH)
    chembl_search = load_json(CHEMBL_SEARCH_PATH)

    pubchem_item = first_item(
        pubchem_properties.get("PropertyTable", {}).get("Properties")
    )
    description_item = first_item(
        pubchem_description.get("InformationList", {}).get("Information")
    )
    chembl_item = first_item(chembl_search.get("molecules"))
    chembl_properties = chembl_item.get("molecule_properties", {})

    if not isinstance(chembl_properties, dict):
        chembl_properties = {}

    return ChemicalIdentity(
        cid=int(pubchem_item.get("CID", 0)),
        title=str(description_item.get("Title", "")),
        chembl_id=str(chembl_item.get("molecule_chembl_id", "")),
        formula=str(pubchem_item.get("MolecularFormula", "")),
        chembl_formula=str(chembl_properties.get("full_molformula", "")),
        molecular_weight=str(pubchem_item.get("MolecularWeight", "")),
        inchikey=str(pubchem_item.get("InChIKey", "")),
    )


def summarize_activity() -> ActivitySummary:
    """Summarize beta1 activity and beta2 selectivity-floor evidence."""
    activity_data = load_json(CHEMBL_ACTIVITIES_PATH)
    beta1_values: list[float] = []
    beta2_floor_values: list[float] = []

    for activity in activity_data.get("activities", []):
        if not isinstance(activity, dict):
            continue

        target = str(activity.get("target_pref_name", "")).lower()
        relation = str(activity.get("standard_relation", ""))
        units = str(activity.get("standard_units", ""))
        standard_type = str(activity.get("standard_type", ""))
        value = as_float(activity.get("standard_value"))

        if value is None or units != "nM":
            continue

        is_beta1 = "beta-1" in target or "beta1" in target
        is_beta2 = "beta2" in target
        is_activity_type = standard_type in {"EC50", "Kd"}

        if is_beta1 and relation == "=" and is_activity_type:
            beta1_values.append(value)
            continue

        if is_beta2 and relation == ">" and standard_type == "EC50":
            beta2_floor_values.append(value)

    return ActivitySummary(
        beta1_nanometer_values=tuple(sorted(beta1_values)),
        beta2_floor_values=tuple(sorted(beta2_floor_values)),
    )


def summarize_literature() -> LiteratureSummary:
    """Extract literature anchors from PubMed snapshots and XML text."""
    beta1_snapshot = load_json(PUBMED_BETA1_PATH)
    hbf_snapshot = load_json(PUBMED_HBF_REFRESH_PATH)
    abstract_text = " ".join(AMPKB1_ABSTRACT_PATH.read_text().split())
    lower_abstract = abstract_text.lower()

    pmids = {
        str(result.get("pmid", ""))
        for result in beta1_snapshot.get("results", [])
        if isinstance(result, dict)
    }
    pmids.update(
        str(result.get("pmid", ""))
        for result in hbf_snapshot.get("results", [])
        if isinstance(result, dict)
    )

    root = ET.fromstring(AMPKB1_ABSTRACT_PATH.read_text())
    xml_text = " ".join(text for text in root.itertext())
    lower_xml = xml_text.lower()

    return LiteratureSummary(
        has_hbf_bridge="41259521" in pmids and "fetal hemoglobin" in lower_abstract,
        has_nrf2_bridge="nrf2" in lower_xml,
        has_ulk1_p62_bridge="ulk1" in lower_xml
        and ("sqstm1" in lower_xml or "p62" in lower_xml),
        has_ugt1a1_caution="30194276" in pmids,
        pmids=tuple(sorted(pmid for pmid in pmids if pmid)),
    )

## Gate Calculation

The decision rule is simple and intentionally strict:

- pass material identity only if PubChem and ChEMBL agree on the formula and known identifiers are present;
- pass target support only if beta1-complex activity is nanomolar while beta2-complex activity is weak in ChEMBL;
- pass mechanism bridge only if the HbF / NRF2 / ULK1-p62 source is present;
- keep metabolism and patient-use cautions visible even when the research-probe gate passes.

In [4]:
identity = build_identity()
activity = summarize_activity()
literature = summarize_literature()

identity_verified = all(
    (
        identity.cid == 71748255,
        identity.title == "PF-06409577",
        identity.chembl_id == "CHEMBL3891221",
        identity.formula == "C19H16ClNO3",
        identity.chembl_formula == identity.formula,
        identity.inchikey == "FHQXLWCFSUSXBF-UHFFFAOYSA-N",
    )
)

result = MaterialGateResult(
    identity_verified=identity_verified,
    beta1_selectivity_supported=(
        activity.has_beta1_activity and activity.has_beta2_selectivity_floor
    ),
    hbf_mechanism_bridge_supported=(
        literature.has_hbf_bridge
        and literature.has_nrf2_bridge
        and literature.has_ulk1_p62_bridge
    ),
    metabolism_caution_present=literature.has_ugt1a1_caution,
    decision="quote_addendum_ready_as_research_probe",
    boundary="not_patient_candidate_not_treatment_advice",
)

summary = {
    "identity": identity.__dict__,
    "activity": {
        "beta1_nanometer_values": activity.beta1_nanometer_values,
        "beta2_floor_values": activity.beta2_floor_values,
    },
    "literature": literature.__dict__,
    "result": result.__dict__,
}

print(json.dumps(summary, indent=2, sort_keys=True))

{
  "activity": {
    "beta1_nanometer_values": [
      3.3,
      7.0,
      8.2,
      9.0,
      28.2
    ],
    "beta2_floor_values": [
      40000.0,
      40000.0,
      40000.0
    ]
  },
  "identity": {
    "chembl_formula": "C19H16ClNO3",
    "chembl_id": "CHEMBL3891221",
    "cid": 71748255,
    "formula": "C19H16ClNO3",
    "inchikey": "FHQXLWCFSUSXBF-UHFFFAOYSA-N",
    "molecular_weight": "341.8",
    "title": "PF-06409577"
  },
  "literature": {
    "has_hbf_bridge": true,
    "has_nrf2_bridge": true,
    "has_ugt1a1_caution": true,
    "has_ulk1_p62_bridge": true,
    "pmids": [
      "30194276",
      "35294578",
      "39855028",
      "41259521"
    ]
  },
  "result": {
    "beta1_selectivity_supported": true,
    "boundary": "not_patient_candidate_not_treatment_advice",
    "decision": "quote_addendum_ready_as_research_probe",
    "hbf_mechanism_bridge_supported": true,
    "identity_verified": true,
    "metabolism_caution_present": true
  }
}


## Interpretation

PF-06409577 passes as a named `AMPKB1` research probe for a qualified-lab quote addendum because identity, beta1-selective activity, and the HbF / noncanonical `NRF2` / `ULK1` / p62 bridge are source-backed.

It does **not** pass as a patient candidate. The useful next action is to ask a lab whether it can source exact material with CoA / purity / lot documentation and run the dual HbF plus alpha-globin/autophagy panel. Generic AMPK activators, metformin-only framing, broad antioxidant framing, K562-only evidence, or missing hemolysis and alpha-globin readouts remain hold/reject conditions.